# 050. Daty i Szeregi Czasowe (Time Series)

Parsowanie znaczników czasu z `errors='coerce'`, akcesor `.dt`, strefy czasowe, kalendarze biznesowe oraz agregacje czasowe (`resample`, `rolling`).


## 1. Bezpieczne parsowanie dat biznesowych z `errors='coerce'`


In [ ]:
import pandas as pd
import numpy as np


In [ ]:

daty_raw = ['01/05/2024 14:30', '15/06/2024 09:15', '31/12/2023', 'USZKODZONY_WPIS', '2024-07-01']

# errors='coerce' zamienia śmieci na NaT zamiast wywalać program
daty_czyste = pd.to_datetime(daty_raw, errors='coerce', dayfirst=True)
print("Sparsowane daty:\n", daty_czyste)


In [ ]:

df_daty = pd.DataFrame({'Data': daty_czyste}).dropna()
df_daty['Rok'] = df_daty['Data'].dt.year
df_daty['Miesiac'] = df_daty['Data'].dt.month
df_daty['DzienTygodnia'] = df_daty['Data'].dt.day_name(locale='pl_PL')
df_daty['CzyWeekend'] = df_daty['Data'].dt.dayofweek >= 5
df_daty


## 2. Resampling i Downsampling szeregów czasowych (`resample`)


In [ ]:
np.random.seed(42)
os_godzinowa = pd.date_range('2024-01-01', periods=720, freq='h') # 30 dni co godzinę
telemetria = pd.DataFrame({
    'Zuzycie_Pradu': np.random.uniform(5.0, 25.0, size=len(os_godzinowa)),
    'Temperatura': 18 + 5 * np.sin(np.linspace(0, 10, len(os_godzinowa)))
}, index=os_godzinowa)


In [ ]:

# Downsampling: z godzin do podsumowań dziennych ('D')
dzienne = telemetria.resample('D').agg({
    'Zuzycie_Pradu': 'sum',
    'Temperatura': ['mean', 'max']
})
dzienne.head()


## 3. Okna kroczące (`rolling`) i pasma zmienności


In [ ]:
ceny_akcji = pd.Series(
    100 + np.cumsum(np.random.normal(0.2, 1.5, 100)),
    index=pd.date_range('2024-01-01', periods=100, freq='D')
)


In [ ]:

df_kursy = pd.DataFrame({'Kurs': ceny_akcji})
df_kursy['SMA_14'] = df_kursy['Kurs'].rolling(window=14, min_periods=5).mean()
df_kursy['Gorne_Pasmo'] = df_kursy['SMA_14'] + 2 * df_kursy['Kurs'].rolling(14).std()
df_kursy['Dolne_Pasmo'] = df_kursy['SMA_14'] - 2 * df_kursy['Kurs'].rolling(14).std()
df_kursy.tail()
